# 🔬 DermaFusion-AI — Kaggle Training Notebook
## EVA-02 Large + ConvNeXt V2 | Dual-Branch Fusion | SOTA 2026

### ⚙️ Before Running:
1. Click **'Add Data'** (right panel) and attach these 4 datasets:
   - `skin-cancer-mnist-ham10000`
   - `isic-2019`
   - `siim-isic-melanoma-classification` (ISIC 2020)
   - `isic-2024-challenge` (ISIC 2024)
2. Set **Accelerator → GPU T4 x2** (or P100) in the right panel
3. Set **Persistence → Files** so outputs survive between sessions
4. Run cells **top to bottom**

## Step 1: Clone Your GitHub Repo

In [ ]:
import os

# Clone the project from GitHub
!git clone https://github.com/ai-with-abdullah/DermaFusion-AI.git /kaggle/working/DermaFusion-AI

# Move into project directory
os.chdir('/kaggle/working/DermaFusion-AI')
print('Working directory:', os.getcwd())

## Step 2: Install Dependencies

In [ ]:
!pip install -q timm>=0.9.12 albumentations>=1.3.1 opencv-python-headless scikit-learn scipy tqdm
print('✅ Dependencies installed')

## Step 3: Verify GPU

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('❌ No GPU found — go to Settings > Accelerator > GPU')

## Step 4: Link Kaggle Datasets → Project data/ folder
This creates symlinks so the code finds datasets at the expected paths without copying any files.

In [ ]:
import os

os.makedirs('/kaggle/working/DermaFusion-AI/data', exist_ok=True)

# ── HAM10000 ──────────────────────────────────────────────────────────────────
ham_src = '/kaggle/input/skin-cancer-mnist-ham10000'
ham_dst = '/kaggle/working/DermaFusion-AI/data/ham10000'
if os.path.exists(ham_src) and not os.path.exists(ham_dst):
    os.symlink(ham_src, ham_dst)
    print(f'✅ HAM10000 linked')
elif os.path.exists(ham_dst):
    print(f'✅ HAM10000 already linked')
else:
    print(f'❌ HAM10000 not found at {ham_src} — did you add the dataset?')

# ── ISIC 2019 ─────────────────────────────────────────────────────────────────
isic19_src = '/kaggle/input/isic-2019'
isic19_dst = '/kaggle/working/DermaFusion-AI/data/isic_2019'
if os.path.exists(isic19_src) and not os.path.exists(isic19_dst):
    os.symlink(isic19_src, isic19_dst)
    print(f'✅ ISIC 2019 linked')
elif os.path.exists(isic19_dst):
    print(f'✅ ISIC 2019 already linked')
else:
    print(f'❌ ISIC 2019 not found at {isic19_src} — did you add the dataset?')

# ── ISIC 2020 ─────────────────────────────────────────────────────────────────
isic20_src = '/kaggle/input/siim-isic-melanoma-classification'
isic20_dst = '/kaggle/working/DermaFusion-AI/data/isic_2020'
if os.path.exists(isic20_src) and not os.path.exists(isic20_dst):
    os.symlink(isic20_src, isic20_dst)
    print(f'✅ ISIC 2020 linked')
elif os.path.exists(isic20_dst):
    print(f'✅ ISIC 2020 already linked')
else:
    print(f'❌ ISIC 2020 not found at {isic20_src} — did you add the dataset?')

# ── ISIC 2024 ─────────────────────────────────────────────────────────────────
isic24_src = '/kaggle/input/isic-2024-challenge'
isic24_dst = '/kaggle/working/DermaFusion-AI/data/isic_2024'
if os.path.exists(isic24_src) and not os.path.exists(isic24_dst):
    os.symlink(isic24_src, isic24_dst)
    print(f'✅ ISIC 2024 linked')
elif os.path.exists(isic24_dst):
    print(f'✅ ISIC 2024 already linked')
else:
    print(f'❌ ISIC 2024 not found at {isic24_src} — did you add the dataset?')

print('\n📁 data/ contents:')
os.system('ls /kaggle/working/DermaFusion-AI/data/')

## Step 5: Phase 1 — Train Segmentation Model (25 epochs)
Trains the **Swin-UNet** to generate lesion masks. Must be done before classifier training.

In [ ]:
import os
os.chdir('/kaggle/working/DermaFusion-AI')

!python train_segmentation.py 2>&1 | tee /kaggle/working/train_segmentation.log
print('\n✅ Segmentation training complete!')

## Step 6: Phase 2 — Train Classifier (25 epochs)
Trains the **EVA-02 Large + ConvNeXt V2** dual-branch fusion classifier.

In [ ]:
import os
os.chdir('/kaggle/working/DermaFusion-AI')

!python train_classifier.py 2>&1 | tee /kaggle/working/train_classifier.log
print('\n✅ Classifier training complete!')

## Step 7: Save Weights to Kaggle Output

In [ ]:
import shutil, os

# Kaggle output directory — files here are saved permanently
output_dir = '/kaggle/working/'

weights_src = '/kaggle/working/DermaFusion-AI/outputs/weights/'
for f in os.listdir(weights_src):
    if f.endswith('.pth'):
        shutil.copy(os.path.join(weights_src, f), output_dir)
        print(f'✅ Saved: {f}')

print('\n📦 All weights saved to Kaggle output. Download from the Output tab.')

## Step 8: Full Evaluation (Metrics + GradCAM++ XAI)

In [ ]:
import os
os.chdir('/kaggle/working/DermaFusion-AI')

!python evaluate.py 2>&1 | tee /kaggle/working/evaluate.log
print('\n✅ Evaluation complete! Check the Output tab for results.')